# SPÉ 1 — Notebook de démarrage : métastases dans des images histopathologiques

**MBA 2 Data & BI — Epitech — 2026-2027**

Ce notebook vous donne une base qui fonctionne de bout en bout, à reprendre et à améliorer :

1. charger le sous-échantillon de Camelyon17-WILDS (39 000 patchs) ;
2. explorer les données par hôpital ;
3. entraîner une **référence simple** (histogrammes de couleur + régression logistique) ;
4. entraîner un **CNN pré-entraîné** (ResNet18) par apprentissage par transfert ;
5. évaluer **hôpital par hôpital** ;
6. regarder ce que voit le modèle avec **Grad-CAM** ;
7. remonter du patch à la lame et au **stade pN**.

**Avant de commencer** (Colab) : menu *Exécution → Modifier le type d'exécution → GPU (T4)*.

Ce notebook est un point de départ, pas une solution : chaque choix (taille d'image, nombre d'époques, seuil, agrégation…) est à questionner et à justifier dans votre rapport.

## 0. Paramètres

In [ ]:
# Où trouver le sous-échantillon ?
#  "drive" : le fichier camelyon17_sous_echantillon.parquet a été copié dans votre Google Drive (le plus rapide)
#  "hf"    : reconstruction automatique depuis Hugging Face (environ 10 Go téléchargés, 10 à 20 minutes sur Colab)
SOURCE = "hf"
DRIVE_PATH = "/content/drive/MyDrive/SPE1/camelyon17_sous_echantillon.parquet"

# Fichiers du pack élèves, à déposer à côté du notebook (ou dans Colab : panneau Fichiers → Importer)
IDS_CSV = "data/ids_sous_echantillon.csv"     # liste des patchs du sous-échantillon
META_ZIP = "data/metadata_complete.zip"       # métadonnées des 455 954 patchs (sans images)
STAGES_CSV = "data/stages.csv"                # classe de chaque lame et stade pN des 100 patientes

# Colab n'ouvre que le notebook : si data/ est absent, on clone le dépôt et on se place dedans
import os
if not os.path.exists("data"):
    if not os.path.exists("model-cancer-sein"):
        os.system("git clone -q https://github.com/Cloud-io09/model-cancer-sein.git")
    os.chdir("model-cancer-sein")

QUICK = False        # True = petit essai rapide (quelques centaines de patchs) pour vérifier que tout marche
EPOCHS_HEAD = 2      # époques avec seule la dernière couche entraînée
EPOCHS_FT = 2        # époques d'ajustement fin (dernier bloc + dernière couche)
SEED = 42

## 1. Installation et imports

In [2]:
import os, io, json, time, random, zipfile, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score, confusion_matrix

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", DEVICE)
HOPITAL = {0: "H0 (entr.)", 1: "H1 (valid. hors distrib.)", 2: "H2 (test, jamais vu)", 3: "H3 (entr.)", 4: "H4 (entr.)"}

Calcul sur : cpu


## 2. Charger le sous-échantillon

Le sous-échantillon contient 39 000 patchs de 96 × 96 pixels, tirés au hasard et **équilibrés par hôpital et par classe** :

| Jeu | Hôpitaux | Patchs | Rôle |
|---|---|---|---|
| `train` | 0, 3, 4 | 24 000 | entraînement |
| `id_val` | 0, 3, 4 | 3 000 | validation « même distribution » (mêmes hôpitaux que l'entraînement) |
| `val_ood` | 1 | 4 000 | validation « hors distribution » : réglages et choix du modèle |
| `test` | 2 | 8 000 | **test final, à n'utiliser qu'une fois** |

Attention : `id_val` vient des mêmes lames que `train`. C'est le découpage officiel de WILDS, mais il surestime la performance. Pour vos propres validations internes, découpez **par lame**.

In [ ]:
def reconstruire_depuis_hf(ids_csv, sortie="camelyon17_sous_echantillon.parquet"):
    """Télécharge les fichiers Parquet de Hugging Face un par un, garde les patchs du sous-échantillon, efface le reste."""
    import pyarrow.parquet as pq, pyarrow as pa
    ids = pd.read_csv(ids_csv)
    voulu = dict(zip(ids.image_id, ids.split))
    api = "https://huggingface.co/api/datasets/wltjr1007/Camelyon17-WILDS/parquet/default"
    shards = json.load(urllib.request.urlopen(api))
    morceaux = []
    for split, urls in shards.items():
        for i, url in enumerate(urls):
            t0 = time.time(); f = f"tmp_{split}_{i}.parquet"
            urllib.request.urlretrieve(url, f)
            t = pq.read_table(f)
            ids_shard = t.column("image_id").to_numpy()
            garder = np.flatnonzero(np.isin(ids_shard, list(voulu)))
            if len(garder):
                s = t.take(pa.array(garder)).to_pandas()
                s["image"] = s["image"].apply(lambda d: d["bytes"])
                morceaux.append(s)
            os.remove(f)
            print(f"{split} {i + 1}/{len(urls)} : {len(garder)} patchs gardés ({time.time() - t0:.0f} s)")
    df = pd.concat(morceaux, ignore_index=True)
    df["split"] = df.image_id.map(voulu)
    df.to_parquet(sortie, index=False)
    return sortie

if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    chemin = DRIVE_PATH
else:
    chemin = "camelyon17_sous_echantillon.parquet"
    if not os.path.exists(chemin):
        chemin = reconstruire_depuis_hf(IDS_CSV, chemin)

df = pd.read_parquet(chemin)
if QUICK:
    df = df.sample(frac=1, random_state=SEED).groupby(["split", "center", "label"]).head(40)
df = df.reset_index(drop=True)
print(len(df), "patchs")
df.groupby(["split", "center", "label"]).size().unstack()

FileNotFoundError: [Errno 2] No such file or directory: 'ids_sous_echantillon.csv'

## 3. Exploration

In [ ]:
def ouvrir(b):
    return Image.open(io.BytesIO(b)).convert("RGB")

fig, axes = plt.subplots(5, 8, figsize=(14, 9))
for r, h in enumerate(range(5)):
    for c in range(8):
        lab = c // 4
        s = df[(df.center == h) & (df.label == lab)].sample(1, random_state=c)
        axes[r, c].imshow(ouvrir(s.image.iloc[0])); axes[r, c].axis("off")
        if c in (0, 4):
            axes[r, c].set_title(f"{HOPITAL[h]}\n{'tumoral' if lab else 'normal'}", fontsize=8)
plt.suptitle("4 patchs normaux puis 4 patchs tumoraux, pour chaque hôpital"); plt.tight_layout(); plt.show()

In [ ]:
# La couleur change-t-elle d'un hôpital à l'autre ? Moyenne des canaux RGB par patch
def moyenne_rgb(b):
    return np.asarray(ouvrir(b), dtype=np.float32).reshape(-1, 3).mean(0)

ech = df.sample(frac=1, random_state=SEED).groupby("center").head(300)
rgb = np.stack(ech.image.map(moyenne_rgb).values)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for k, canal in enumerate(["Rouge", "Vert", "Bleu"]):
    for h in range(5):
        axes[k].hist(rgb[ech.center.values == h, k], bins=30, alpha=.45, label=HOPITAL[h], density=True)
    axes[k].set_title(canal)
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 4. Référence simple : histogrammes de couleur + régression logistique

Un modèle complexe ne vaut que s'il bat une référence simple. Ici : 16 classes par canal RGB, soit 48 variables par patch.

In [ ]:
def hist_couleur(b, bins=16):
    a = np.asarray(ouvrir(b))
    return np.concatenate([np.histogram(a[..., k], bins=bins, range=(0, 255), density=True)[0] for k in range(3)])

X = {s: np.stack(df[df.split == s].image.map(hist_couleur).values) for s in ["train", "id_val", "val_ood"]}
y = {s: df[df.split == s].label.values for s in ["train", "id_val", "val_ood"]}
ref = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X["train"], y["train"])
for s in ["id_val", "val_ood"]:
    p = ref.predict_proba(X[s])[:, 1]
    print(f"{s:8s} ROC-AUC = {roc_auc_score(y[s], p):.3f}   rappel = {recall_score(y[s], p > .5):.3f}")

## 5. CNN pré-entraîné par transfert (ResNet18)

Les patchs font 96 × 96. On les garde à cette taille pour aller vite ; les agrandir à 224 × 224 est une piste à tester et à justifier.

In [ ]:
MOY, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # normalisation ImageNet
T = torchvision.transforms
tf_entr = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.ToTensor(), T.Normalize(MOY, STD)])
tf_eval = T.Compose([T.ToTensor(), T.Normalize(MOY, STD)])

class Patchs(Dataset):
    def __init__(self, d, tf):
        self.b, self.y, self.tf = d.image.values, d.label.values, tf
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.tf(ouvrir(self.b[i])), int(self.y[i])

def chargeur(split, tf, shuffle):
    return DataLoader(Patchs(df[df.split == split], tf), batch_size=128, shuffle=shuffle,
                      num_workers=2, pin_memory=(DEVICE == "cuda"))

dl_train = chargeur("train", tf_entr, True)

modele = torchvision.models.resnet18(weights="IMAGENET1K_V1")
modele.fc = nn.Linear(modele.fc.in_features, 2)
modele = modele.to(DEVICE)

@torch.no_grad()
def probas(m, dl):
    m.eval(); out = []
    for x, _ in dl:
        out.append(torch.softmax(m(x.to(DEVICE)), 1)[:, 1].cpu())
    return torch.cat(out).numpy()

def entrainer(m, params, epochs, lr):
    opt = torch.optim.Adam(params, lr=lr); crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train(); t0 = time.time(); perte = 0
        for x, yy in dl_train:
            x, yy = x.to(DEVICE), yy.to(DEVICE)
            opt.zero_grad(); l = crit(m(x), yy); l.backward(); opt.step(); perte += l.item() * len(yy)
        p = probas(m, chargeur("val_ood", tf_eval, False))
        auc = roc_auc_score(df[df.split == "val_ood"].label, p)
        print(f"époque {ep + 1}/{epochs}  perte {perte / len(dl_train.dataset):.3f}  ROC-AUC val_ood {auc:.3f}  ({time.time() - t0:.0f} s)")

# Étape 1 : seule la dernière couche apprend
for p in modele.parameters():
    p.requires_grad = False
for p in modele.fc.parameters():
    p.requires_grad = True
entrainer(modele, modele.fc.parameters(), EPOCHS_HEAD, 1e-3)

# Étape 2 : ajustement fin du dernier bloc
for p in modele.layer4.parameters():
    p.requires_grad = True
entrainer(modele, list(modele.layer4.parameters()) + list(modele.fc.parameters()), EPOCHS_FT, 1e-4)
torch.save(modele.state_dict(), "resnet18_spe1.pt")

## 6. Évaluation hôpital par hôpital

Le seuil de 0,5 n'est qu'un point de départ. **Un faux négatif est une métastase manquée** : choisissez votre seuil sur la validation en fonction du rappel visé, jamais sur le test.

In [ ]:
def evaluer(d, p, seuil=0.5):
    lignes = []
    for h, g in d.assign(p=p).groupby("center"):
        pred = g.p >= seuil
        lignes.append({"hôpital": HOPITAL[h], "n": len(g), "ROC-AUC": roc_auc_score(g.label, g.p),
                       "rappel": recall_score(g.label, pred), "précision": precision_score(g.label, pred, zero_division=0),
                       "accuracy": accuracy_score(g.label, pred)})
    return pd.DataFrame(lignes).round(3)

for s in ["id_val", "val_ood"]:
    d = df[df.split == s].reset_index(drop=True)
    print(s); display(evaluer(d, probas(modele, chargeur(s, tf_eval, False))))

# Seuil choisi sur val_ood pour viser un rappel d'au moins 95 %
d_v = df[df.split == "val_ood"].reset_index(drop=True); p_v = probas(modele, chargeur("val_ood", tf_eval, False))
seuils = np.linspace(0.01, 0.99, 99)
rappels = np.array([recall_score(d_v.label, p_v >= s) for s in seuils])
SEUIL = seuils[rappels >= 0.95].max() if (rappels >= 0.95).any() else 0.5
print("Seuil retenu :", round(SEUIL, 2))

In [ ]:
# TEST FINAL — hôpital 2, jamais vu. À n'exécuter qu'une fois, quand tous vos choix sont faits.
LANCER_TEST = False
if LANCER_TEST:
    d_t = df[df.split == "test"].reset_index(drop=True); p_t = probas(modele, chargeur("test", tf_eval, False))
    display(evaluer(d_t, p_t, SEUIL))
    print(confusion_matrix(d_t.label, p_t >= SEUIL))

## 7. Que regarde le modèle ? Grad-CAM

Grad-CAM colore les zones du patch qui ont le plus pesé dans la décision « tumoral ». Regardez des patchs bien et mal classés, de plusieurs hôpitaux : le modèle regarde-t-il les noyaux des cellules, ou la teinte générale ?

In [ ]:
def gradcam(m, x):
    acts = {}
    h = m.layer4.register_forward_hook(lambda mod, i, o: acts.__setitem__("a", o))
    m.eval()
    out = m(x.unsqueeze(0).to(DEVICE)); h.remove()
    g = torch.autograd.grad(out[0, 1], acts["a"])[0]
    w = g.mean((2, 3), keepdim=True)
    cam = torch.relu((w * acts["a"]).sum(1)).squeeze().detach().cpu().numpy()
    cam = np.array(Image.fromarray(cam / (cam.max() + 1e-8)).resize((96, 96), Image.BILINEAR))
    return cam, torch.softmax(out, 1)[0, 1].item()

for p in modele.parameters():
    p.requires_grad = True
ex = df[df.split == "val_ood"].sample(frac=1, random_state=1).groupby("label").head(4).sort_values("label")
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for k, (_, r) in enumerate(ex.iterrows()):
    img = ouvrir(r.image); cam, p = gradcam(modele, tf_eval(img))
    axes[0, k].imshow(img); axes[0, k].set_title(f"vrai {r.label} / p={p:.2f}", fontsize=8)
    axes[1, k].imshow(img); axes[1, k].imshow(cam, cmap="jet", alpha=.45)
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.tight_layout(); plt.show()

## 8. Du patch à la lame, puis à la patiente

Les métadonnées complètes donnent la position de chacun des 455 954 patchs dans sa lame. On peut donc dessiner la **carte des patchs** d'une lame, avec la vérité et, sur les patchs du sous-échantillon, les prédictions.

Rappel : les 50 lames de WILDS sont toutes positives, et aucune patiente n'a ses 5 ganglions annotés. Le niveau patiente se travaille donc en deux temps : vérifier les règles pN sur les classes officielles, puis mesurer l'effet de vos erreurs.

In [ ]:
with zipfile.ZipFile(META_ZIP) as z:
    meta = pd.read_csv(z.open("metadata_complete.csv"))
stages = pd.read_csv(STAGES_CSV)
lames = meta.groupby("slide").agg(patient=("patient", "first"), node=("node", "first"), center=("center", "first"),
                                   n_patchs=("label", "size"), part_tumorale=("label", "mean")).reset_index()
lames["nom"] = [f"patient_{p:03d}_node_{n}.tif" for p, n in zip(lames.patient, lames.node)]
lames = lames.merge(stages.rename(columns={"patient": "nom", "stage": "classe_officielle"})[["nom", "classe_officielle"]], on="nom")
lames.head(10)

In [ ]:
# Carte d'une lame : vérité (tous les patchs) et prédictions (patchs de cette lame présents dans le sous-échantillon)
LAME = int(lames.sort_values("part_tumorale", ascending=False).slide.iloc[0])
v = meta[meta.slide == LAME]
s = df[df.slide == LAME].reset_index(drop=True)
p_s = probas(modele, DataLoader(Patchs(s, tf_eval), batch_size=128)) if len(s) else []
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
axes[0].scatter(v.x_coord, v.y_coord, c=v.label, s=2, cmap="coolwarm"); axes[0].set_title(f"Lame {LAME} : vérité (rouge = tumoral)")
if len(s):
    im = axes[1].scatter(s.x_coord, s.y_coord, c=p_s, s=6, cmap="coolwarm", vmin=0, vmax=1); plt.colorbar(im, ax=axes[1])
axes[1].set_title("Probabilité prédite (patchs du sous-échantillon)")
axes[0].invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
# Règles pN simplifiées de CAMELYON17 (5 ganglions par patiente)
def stade_pN(classes):
    classes = list(classes)
    macro, micro, itc = classes.count("macro"), classes.count("micro"), classes.count("itc")
    atteints = macro + micro                     # les cellules isolées ne comptent pas
    if atteints == 0:
        return "pN0(i+)" if itc else "pN0"
    if macro == 0:
        return "pN1mi"
    return "pN1" if atteints <= 3 else "pN2"

gl = stages[stages.patient.str.endswith(".tif")].copy(); gl["pid"] = gl.patient.str[:11]
pt = stages[stages.patient.str.endswith(".zip")].copy(); pt["pid"] = pt.patient.str[:11]
recalcule = gl.groupby("pid").stage.apply(stade_pN)
print("Concordance des règles avec le stade officiel :", (recalcule == pt.set_index("pid").stage).mean())

# À vous : prédire une classe (itc / micro / macro) pour chaque lame annotée à partir de vos prédictions,
# remplacer la classe officielle de ces lames dans `gl`, recalculer le stade pN et compter les changements.

## 9. Et maintenant

- Découper vos propres validations **par lame**, et tester plusieurs graines.
- Gérer la variabilité de couleur : augmentation de couleur (`ColorJitter`), normalisation de la coloration, niveaux de gris… et mesurer l'effet **par hôpital**.
- Comparer d'autres architectures ou une taille d'entrée de 224 × 224.
- Estimer l'étendue des métastases par lame, en déduire une classe, et mesurer l'effet sur le stade pN.
- Discuter : faux négatifs, rôle du pathologiste, jeu équilibré contre réalité clinique, cadre réglementaire.

Sources : CAMELYON17 (CC0), https://camelyon17.grand-challenge.org ; Camelyon17-WILDS, Koh et al., ICML 2021 ; miroir Hugging Face `wltjr1007/Camelyon17-WILDS`.